In [ ]:
import io
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import numpy as np

In [ ]:
import os
import shutil

# Make sure the .kaggle folder exists
os.makedirs("/root/.kaggle", exist_ok=True)

# Move kaggle.json from /content to /root/.kaggle
shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")

# Set correct permissions
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully!")


Kaggle API configured successfully!


In [ ]:
!pip install kaggle
!kaggle datasets download -d gunavenkatdoddi/eye-diseases-classification

Dataset URL: https://www.kaggle.com/datasets/gunavenkatdoddi/eye-diseases-classification
License(s): ODbL-1.0
 98% 721M/736M [00:02<00:00, 263MB/s]
100% 736M/736M [00:02<00:00, 359MB/s]


In [ ]:
!unzip -q eye-diseases-classification.zip -d /content/eye_diseases_dataset

In [ ]:
%%writefile app.py
import os
import io
import numpy as np
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from PIL import Image
import torch
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load class names ===
DATA_DIR = "/content/eye_diseases_dataset/dataset"
classes = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

# === Load model ===
num_classes = len(classes)
model = timm.create_model("efficientnet_b3", pretrained=False, num_classes=num_classes)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()

# === Preprocessing ===
img_size = 384
transform = A.Compose([
    A.Resize(img_size, img_size),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

app = FastAPI(title="Eye Disease Prediction API")

@app.get("/")
def home():
    return {"status": "API is running"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    try:
        content = await file.read()
        img = Image.open(io.BytesIO(content)).convert("RGB")
        img = np.array(img)

        img = transform(image=img)["image"].unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = model(img)
            probs = F.softmax(outputs, dim=1)
            conf, pred = torch.max(probs, 1)
            predicted_class = idx_to_class[int(pred.item())]

        return {
            "predicted_class": predicted_class,
            "confidence": float(conf.item())
        }

    except Exception as e:
        return JSONResponse(content={"error": str(e)}, status_code=500)


Writing app.py


In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 &


nohup: appending output to 'nohup.out'


In [ ]:
!sleep 5
!tail -n 20 nohup.out


In [ ]:
!pip install pyngrok


In [ ]:
!ls -l


total 795656
-rw-r--r-- 1 root root      1869 Dec  7 14:04 app.py
-rw-r--r-- 1 root root  43360463 Dec  7 13:55 best_model.pth
-rw-r--r-- 1 root root 771355331 Aug 28  2022 eye-diseases-classification.zip
drwxr-xr-x 3 root root      4096 Dec  7 14:02 eye_diseases_dataset
-rw------- 1 root root       195 Dec  7 14:05 nohup.out
drwxr-xr-x 2 root root      4096 Dec  7 14:05 __pycache__
drwxr-xr-x 1 root root      4096 Nov 20 14:30 sample_data


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("2tMgDnAs5iUS2kpaAcplC77kifS_69dTiKTPPrqMYeqQ4LKc8")
public_url = ngrok.connect(8000)
public_url


<NgrokTunnel: "https://2007fcf967b7.ngrok-free.app" -> "http://localhost:8000">

In [ ]:
import nest_asyncio
nest_asyncio.apply()

!uvicorn app:app --host 0.0.0.0 --port 8000


INFO:     Started server process [11371]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [ ]:
!pip install pyngrok

In [ ]:
!uvicorn app:app --host 0.0.0.0 --port 8000 --reload


INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [3776] using StatReload
ERROR:    Error loading ASGI app. Could not import module "app".
INFO:     Stopping reloader process [3776]


In [ ]:
from pyngrok import ngrok
import nest_asyncio

ngrok.set_auth_token("2tMgDnAs5iUS2kpaAcplC77kifS_69dTiKTPPrqMYeqQ4LKc8")

public_url = ngrok.connect(8000)
print("🔗 Public URL:", public_url)

nest_asyncio.apply()


🔗 Public URL: NgrokTunnel: "https://c1cdea57a9f5.ngrok-free.app" -> "http://localhost:8000"


In [ ]:

!uvicorn app:app --host 0.0.0.0 --port 8000 --reload


INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5327] using StatReload
ERROR:    Error loading ASGI app. Could not import module "app".


INFO:     Stopping reloader process [5327]


/usr/lib/python3.12/threading.py:299: RuntimeWarning: coroutine 'Server.serve' was never awaited
  def __enter__(self):


In [ ]:
import requests

url = "YOUR_NGROK_URL/predict"
files = {"file": open("test_image.jpg", "rb")}
res = requests.post(url, files=files)
print(res.json())
